# Part 5 — Development and Model Selection

**BFT65B — Big Data Analytics in Finance | Student ID: 11230945**

Compare predefined models using purged chronological development folds. Set an event-level economic alarm threshold from out-of-fold predictions, demonstrate Spark MLlib and seal the final development estimators. No historical test is scored in this notebook.

## 1. Repository and runtime
Use a fresh CPU runtime, Java 17 or later, and the pinned dependencies. Use the same Python and dependency versions for both Part 5 notebooks.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
java = shutil.which("java")
if java is None:
    raise RuntimeError("Java 17 or later is required")
os.environ["JAVA_HOME"] = str(Path(java).resolve().parents[1])
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
print(subprocess.run([java,"-version"],check=True,capture_output=True,text=True).stderr)
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"requirements.txt")],check=True)
sys.path.insert(0,str(REPO/'src'))
def command(action):
    proc = subprocess.Popen([sys.executable,str(REPO/'run_part5.py'),action,'--workspace',str(WORKSPACE)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='',flush=True)
    if proc.wait()!=0:
        raise RuntimeError(f'Part 5 {action} failed; inspect diagnostics')

## 2. Mandatory offline validation
Run the complete test suite, including real Parquet and Spark integration. A failed or skipped test stops the pipeline.

In [ ]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path('/content/part5_tests.xml')
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable,'-m','pytest','-q','-ra','--tb=short','--show-capture=no',f'--junitxml={result_path}','tests'],cwd=REPO,env=dict(os.environ,PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',SPARK_LOCAL_IP='127.0.0.1',PYSPARK_PYTHON=sys.executable,MPLBACKEND='Agg'),capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if not result_path.is_file(): raise RuntimeError('Missing pytest report')
cases=list(ET.parse(result_path).getroot().iter('testcase'))
problems=[(case,node) for case in cases for node in case if node.tag in ('failure','error','skipped')]
for case,node in problems:
    print(case.get('classname'),case.get('name'),node.tag,node.get('message'),node.text or '')
if result.returncode or not cases or problems: raise RuntimeError('Offline validation failed; inspect diagnostics')
TESTS_PASSED=True
print(f'Mandatory offline validation: PASS ({len(cases)} tests, zero skips)')

## 3. Verified upstream identities
Check the sealed Gold and Silver inventories and Bronze manifest identity. Development does not decode the historical test partitions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORKSPACE=Path('/content/drive/MyDrive/11230945')
assert TESTS_PASSED
from fdic_ews.modelling.pipeline import upstream, evidence_path, verify_development, verify_oof
from fdic_ews.contracts import sha256
GOLD, gm, SILVER, IDENTITY = upstream(WORKSPACE)
EVIDENCE=evidence_path(WORKSPACE,IDENTITY)
ATTEMPT=EVIDENCE.parent/'test_attempt.json'
print('Verified upstream identities:',json.dumps(IDENTITY,indent=2))
print('Gold decision quarters:',gm['summary']['decision_quarters'])
print('Gold predictors:',gm['summary']['feature_count'])
print('Represented test-calendar failures:',gm['summary']['test_distinct_failure_certs_in_calendar_period'])
print('Evidence directory:',EVIDENCE)

## 4. Fixed chronological design and candidate specifications
The late validation block is sparse in failures. Four validation blocks cover distinct historical regimes. Every fold has a four-quarter gap between its training end and validation start; model fitting and threshold estimation use development only.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from fdic_ews.modelling.design import FOLDS,SPECS,validate_design
validate_design()
display(pd.DataFrame([{'fold':f.name,'training_end':f.train_end,'validation_start':f.start,'validation_end':f.end} for f in FOLDS]))
display(pd.DataFrame([{'specification':name,'estimator':kind,'raw_predictors':len(features),'capital_zero_as_missing':mask} for name,(kind,features,mask) in SPECS.items()]))

## 5. Execute development and seal the choices
The initial run performs all 32 fold fits, development-only threshold optimization, the native Spark demonstration and two final development refits. An existing sealed run is inspected without refitting. An incomplete run must be diagnosed; do not overwrite a sealed run.

In [ ]:
if (EVIDENCE/'development_manifest.json').is_file():
    print('Existing development seal: read-only verification.')
else:
    if ATTEMPT.exists(): raise RuntimeError('Historical test already attempted; development is closed')
    command('develop')
DM=verify_development(EVIDENCE,IDENTITY)
verify_oof(EVIDENCE,DM)
S=DM['summary']
SELECTED=S['selected_model']
print('Development state:',DM['state'])
print('Selection:',SELECTED)
print('Seal SHA-256:',sha256(EVIDENCE/'development_manifest.json'))
print('Historical test scored: no (this notebook)')

## 6. Chronological validation results
Average precision is the primary selection measure. Report fold-specific prevalence, AP, ROC-AUC and Brier score. Equal-weight fold averages are used for selection; pooled OOF metrics are complementary because score scales can vary across folds.

In [ ]:
fold_counts=pd.read_csv(EVIDENCE/'fold_counts.csv')
fold_metrics=pd.read_csv(EVIDENCE/'fold_metrics.csv')
comparison=pd.read_csv(EVIDENCE/'model_comparison.csv')
pooled=pd.read_csv(EVIDENCE/'pooled_metrics.csv')
print('Table 1. Chronological fold populations'); display(fold_counts)
print('Table 2. Fold-level average precision'); display(fold_metrics.pivot(index='model',columns='fold',values='ap').round(6))
print('Table 3. Model comparison'); display(comparison.round(6))
print('Table 4. Pooled out-of-fold ranking diagnostics'); display(pooled.round(6))
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(figsize=(7.2,3.7)); ordered=comparison.sort_values('mean_ap'); ax.barh(ordered.model,ordered.mean_ap,color=['#365c72' if n==SELECTED else '#a2b3bd' for n in ordered.model]); ax.set_xlabel('Mean fold average precision'); ax.grid(axis='x',alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(7.2,3.5)); pivot=fold_metrics.pivot(index='fold',columns='model',values='ap');
for name in ['prior','logistic','hist_gradient','lightgbm_full']:
    ax.plot(pivot.index,pivot[name],marker='o',label=name)
ax.set(ylabel='Average precision',xlabel='Validation fold'); ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)

## 7. Feature contribution and capital-zero sensitivity
Compare full LightGBM with fixed feature ablations and an alternative zero treatment under identical model settings. These are development comparisons, not test-informed feature selection. Training-fit importance, if available, is descriptive.

In [ ]:
ablation=comparison[comparison.model.str.startswith('lightgbm')].copy()
full=float(ablation.loc[ablation.model.eq('lightgbm_full'),'mean_ap'].iloc[0]); ablation['delta_ap_vs_full']=ablation.mean_ap-full
print('Table 5. Controlled LightGBM specifications'); display(ablation[['model','mean_ap','early_folds_mean_ap','late_fold_ap','delta_ap_vs_full']].round(6))
importance_path=EVIDENCE/'final_model_importance.csv'
if importance_path.is_file():
    print('Selected-model descriptive training-fit importance (top 12)'); display(pd.read_csv(importance_path).head(12))
else:
    print('The selected estimator has no native gain/coefficient diagnostic; use the controlled ablations.')

## 8. Data-based economic scenarios and locked thresholds
Missed failures are charged once per distinct represented CERT–failure-date. False alarms are charged per negative decision row. The primary missed-event cost uses a median dated FDIC resolution-cost proxy when at least 20 eligible events are available; otherwise the predeclared asset-based scenario is used. These are decision-cost assumptions, not measured supervisory savings.

In [ ]:
cost=json.loads((EVIDENCE/'cost_policy.json').read_text())
print('Table 6. Economic proxy provenance'); display(pd.DataFrame([cost]).T.rename(columns={0:'value'}))
sensitivity=pd.read_csv(EVIDENCE/'threshold_sensitivity.csv');
print('Table 7. OOF threshold sensitivity'); display(sensitivity[['model','scenario','c_fp','c_fn','threshold','fp','missed_events','detected_events','event_recall','loss_usd_thousands']].round(6))
print('Table 8. Locked operating policies on OOF scores'); display(pd.read_csv(EVIDENCE/'development_operating.csv').round(6))
print('Frozen thresholds:',S['thresholds'])

## 9. Native Spark MLlib demonstration
Fit a Spark pipeline on a sampled training population from the first fold; evaluate all labelled first-fold validation observations at natural prevalence. Sampling is training-only. This demonstration does not participate in model selection.

In [ ]:
spark=json.loads((EVIDENCE/'spark_demo.json').read_text())
assert not spark['validation_sampled']
print('Table 9. Spark MLlib development-only diagnostics'); display(pd.DataFrame([spark]).T.rename(columns={0:'value'}))

## 10. Sealed development handoff
Confirm that model, features, costs, cutoff and environment are sealed before opening Notebook 07. Save this notebook with its outputs. Do not modify development evidence after sealing.

In [ ]:
print('Selected specification:',SELECTED)
print('Final fit:',json.dumps(json.loads((EVIDENCE/'final_fit.json').read_text()),indent=2))
print('Runtime:',json.dumps(DM['environment'],indent=2))
print('Development observations:',S['development_rows'])
print('Development seal:',sha256(EVIDENCE/'development_manifest.json'))
print('Workspace test attempt already exists:',ATTEMPT.exists())
print('Next: Notebook 07 — Historical Test and Evaluation')